# Avaliação em Logs Reais (Chaos Lab)

Este notebook carrega um modelo já treinado (adaptador LoRA) e realiza a inferência em logs reais do ambiente (`reais/teste_real.jsonl`), sem executar treinamento.

Utilizado para validar o desempenho do modelo em dados não vistos e comparar a evolução das versões.

## Pré-requisitos
- Arquivos `triagem.py` e `preparar_reais.py` na mesma pasta.
- Pasta `runbooks/` com os arquivos `.md`.
- Pasta `logs_raw/` com os logs brutos do lab.

## 1. Preparar as janelas de teste real
Gera o arquivo `reais/teste_real.jsonl` a partir dos logs brutos em `logs_raw/`.

In [ ]:
!python preparar_reais.py

## 2. Carregar o modelo treinado
Carrega o modelo base em 4-bit com os adaptadores salvos localmente. Altere `PASTA_MODELO` para avaliar versões anteriores (ex: `gemma4-e2b-incident-triage-v5`).

In [ ]:
import json
from functools import partial
from pathlib import Path

from datasets import load_dataset
from unsloth import FastModel

import triagem as tri

PASTA_MODELO = "gemma4-e2b-incident-triage-v6"

model, tokenizer = FastModel.from_pretrained(
    model_name=PASTA_MODELO,
    max_seq_length=2048,
    load_in_4bit=True,
    text_only=True,
    device_map={"": 0},
    offload_embedding=True,
)
FastModel.for_inference(model)

# Preparar as funções com o modelo carregado
analisar_logs = partial(tri.analisar_logs, model, tokenizer)
triagem = partial(tri.triagem, model, tokenizer)
avaliar = partial(tri.avaliar, model, tokenizer)


## 3. Avaliação nos logs reais
Compara o output do modelo com os rótulos esperados (ground truth) e exibe as métricas consolidadas.

In [ ]:
reais = load_dataset("json", data_files="reais/teste_real.jsonl")["train"]
resultado = avaliar(reais, len(reais), f"Avaliação: {PASTA_MODELO} (Logs Reais)")

## 4. Inspeção de um caso específico
Exibe o log de entrada, a resposta esperada e a resposta gerada pelo modelo para análise manual. Altere o índice `i` para navegar entre as janelas.

In [ ]:
i = 0
exemplo = reais[i]

# Extrai o log bruto de dentro do prompt formatado
prompt = exemplo["conversations"][0]["content"]
log_bruto = prompt.split("Logs:\n```\n", 1)[-1].rsplit("\n```", 1)[0]

esperado = json.loads(exemplo["conversations"][1]["content"])
previsto = triagem(log_bruto)

print("--- LOG DE ENTRADA ---")
print(log_bruto[-1000:]) # Exibe apenas o final do log para não poluir a tela

print("\n--- ESPERADO ---")
print(json.dumps(esperado, indent=2, ensure_ascii=False))

print("\n--- PREVISTO (Modelo + Guardrail) ---")
if previsto:
    print(json.dumps(previsto, indent=2, ensure_ascii=False))
else:
    print("Falha na geração de JSON válido.")

## 5. Salvar resultados da avaliação
Exporta as métricas para um arquivo JSON na pasta `resultados/`, útil para gerar gráficos ou documentar a evolução do projeto.

In [ ]:
Path("resultados").mkdir(exist_ok=True)

# Gera o nome do arquivo dinamicamente baseado na versão do modelo
nome_arquivo = f"resultados/avaliacao_{PASTA_MODELO}.json"

with open(nome_arquivo, "w", encoding="utf-8") as f:
    json.dump({grupo: dict(valores) for grupo, valores in resultado.items()}, f, indent=2, ensure_ascii=False)

print(f"Métricas salvas com sucesso em: {nome_arquivo}")